In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab Notebooks/AI-Machine Learning/Fake Followers Detection')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import

In [2]:
import kagglehub
import pandas as pd
import os

## Load Dataset

In [3]:

path1 = kagglehub.dataset_download("rajumavinmar/fake-instagram-profile-dataset")

def get_csv_path(folder_path):
    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            return os.path.join(folder_path, file)
    return None

csv1 = get_csv_path(path1)

# 3. Muat ke Pandas DataFrame
df1 = pd.read_csv(csv1) if csv1 else None

Using Colab cache for faster access to the 'fake-instagram-profile-dataset' dataset.


In [4]:
df1.head()

,profile pic,nums/length username,fullname words,nums/length fullname,name==username,description length,external URL,private,#posts,#followers,#follows,fake
0,1,0.27,0,0.0,0,53,0,0,32,1000,955,0
1,1,0.00,2,0.0,0,44,0,0,286,2740,533,0
2,1,0.10,2,0.0,0,0,0,1,13,159,98,0
3,1,0.00,1,0.0,0,82,0,0,679,414,651,0
4,1,0.00,2,0.0,0,0,0,1,6,151,126,0


## Data Preprocessing

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Bersihkan nama kolom (opsional tapi praktik yang baik)
# Mengganti spasi dengan underscore dan menghapus karakter aneh
df1.columns = df1.columns.str.replace(' ', '_').str.replace('#', 'num_').str.replace('==', '_eq_')
print("Kolom setelah dibersihkan:\n", df1.columns.tolist())

# 2. Pisahkan Fitur (X) dan Label (y)
X = df1.drop(columns=['fake'])
y = df1['fake']

# 3. Pisahkan Data Training (80%) dan Data Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Standarisasi Skala (Scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Gunakan transform, bukan fit_transform untuk test set

print(f"\nJumlah data latih: {X_train_scaled.shape[0]} baris")
print(f"Jumlah data uji: {X_test_scaled.shape[0]} baris")
print(f"Jumlah fitur (input): {X_train_scaled.shape[1]}")

Kolom setelah dibersihkan:
 ['profile_pic', 'nums/length_username', 'fullname_words', 'nums/length_fullname', 'name_eq_username', 'description_length', 'external_URL', 'private', 'num_posts', 'num_followers', 'num_follows', 'fake']

Jumlah data latih: 4000 baris
Jumlah data uji: 1000 baris
Jumlah fitur (input): 11


## Arsitektur Model

In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Jumlah input berdasarkan fitur dataset kita (harus 11)
input_dim = X_train_scaled.shape[1]

# Membangun Arsitektur Neural Network
model = Sequential([
    # Layer Input & Hidden Layer Pertama (32 Neuron)
    Dense(32, activation='relu', input_shape=(input_dim,)),
    Dropout(0.2), # Mencegah model menghafal data (regularization)
    
    # Hidden Layer Kedua (16 Neuron)
    Dense(16, activation='relu'),
    Dropout(0.2),
    
    # Layer Output (1 Neuron dengan aktivasi Sigmoid)
    # Sigmoid menekan output menjadi probabilitas antara 0.0 hingga 1.0
    Dense(1, activation='sigmoid') 
])

# Kompilasi Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

# Tampilkan ringkasan arsitektur
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 929 (3.63 KB)

 Trainable params: 929 (3.63 KB)

 Non-trainable params: 0 (0.00 B)

## Training Loop

### Tentukan Nama Model

In [7]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

In [8]:
model_save_path = "fluency_fake_follower_model_v1.keras"

### Buat CallBack

In [9]:
# 2. Buat Callbacks
checkpoint = ModelCheckpoint(
    filepath=model_save_path, 
    monitor='val_accuracy', # Pantau akurasi pada data validasi
    mode='max',             # Kita ingin nilai akurasi yang maksimal
    save_best_only=True,    # HANYA simpan jika memecahkan rekor sebelumnya
    verbose=1               # Beritahu kita jika berhasil menyimpan model baru
)

early_stopping = EarlyStopping(
    monitor='val_loss',     # Pantau tingkat error di data validasi
    patience=10,            # Tunggu 10 epoch. Jika tidak membaik, HENTIKAN training.
    restore_best_weights=True # Kembalikan bobot model ke versi terbaik saat berhenti
)


In [10]:


# 3. Mulai proses belajar dengan Callbacks!
print("Memulai pelatihan model tingkat lanjut...")
history = model.fit(
    X_train_scaled, 
    y_train, 
    epochs=100, # Kita naikkan jadi 100 karena ada EarlyStopping yang akan menghentikannya jika sudah mentok
    batch_size=32, 
    validation_split=0.2, 
    callbacks=[checkpoint, early_stopping], # Masukkan senjata rahasia kita di sini
    verbose=1
)

print(f"\nTraining selesai! Model terbaik Anda telah diamankan di: {model_save_path}")

Memulai pelatihan model tingkat lanjut...
Epoch 1/100
 94/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7766 - loss: 0.5269
Epoch 1: val_accuracy improved from None to 0.93875, saving model to fluency_fake_follower_model_v1.keras

Epoch 1: finished saving model to fluency_fake_follower_model_v1.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8575 - loss: 0.4252 - val_accuracy: 0.9388 - val_loss: 0.2267
Epoch 2/100
 85/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9307 - loss: 0.2420
Epoch 2: val_accuracy improved from 0.93875 to 0.94875, saving model to fluency_fake_follower_model_v1.keras

Epoch 2: finished saving model to fluency_fake_follower_model_v1.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9316 - loss: 0.2166 - val_accuracy: 0.9488 - val_loss: 0.1426
Epoch 3/100
 94/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9396 - loss: 0.1727
Epoch 3: val_accuracy improved from 0.94875 to 0.95500, saving model to fluency_fake_follower_model_v1.k

## Simpan Scaller

In [11]:
import joblib

# Menyimpan 'rumus' scaler ke Google Drive Anda
scaler_filename = "fluency_scaler_v1.pkl"
joblib.dump(scaler, scaler_filename)

print(f"Aman! Scaler berhasil disimpan sebagai: {scaler_filename}")

Aman! Scaler berhasil disimpan sebagai: fluency_scaler_v1.pkl


In [12]:
import numpy as np

# Fungsi simulasi backend Collabo
def cek_fake_follower(data_akun):
    # 1. Kecilkan skala angka menggunakan scaler yang sudah dilatih
    data_scaled = scaler.transform(data_akun)
    
    # 2. Minta AI menebak
    probabilitas = model.predict(data_scaled, verbose=0)[0][0]
    
    # 3. Terjemahkan probabilitas menjadi persentase
    persentase_bot = probabilitas * 100
    
    print(f"Skor Indikasi Bot: {persentase_bot:.2f}%")
    if probabilitas > 0.5:
        print("🚨 KESIMPULAN: Akun ini kemungkinan besar PALSU/BOT!")
    else:
        print("✅ KESIMPULAN: Akun ini kemungkinan besar ASLI (Manusia).")
    print("-" * 50)

# ==========================================
# MARI KITA UJI DENGAN DATA FIKTIF
# Urutan 11 kolom: 
# [profile_pic, nums_length_username, fullname_words, nums_length_fullname, 
#  name_eq_username, description_length, external_URL, private, num_posts, num_followers, num_follows]
# ==========================================

print("Test Kasus 1: Akun Influencer Normal")
# Punya foto(1), username bersih(0), nama 2 kata(2), bio panjang(150), followers 15k, following 500
akun_asli = np.array([[1, 0.0, 2, 0.0, 0, 150, 0, 0, 120, 15000, 500]])
cek_fake_follower(akun_asli)

print("Test Kasus 2: Akun Bot Giveaway/Spam")
# Gak ada foto(0), username penuh angka(0.8), bio kosong(0), followers 10, following 7500
akun_bot = np.array([[0, 0.8, 1, 0.5, 0, 0, 0, 0, 0, 10, 7500]])
cek_fake_follower(akun_bot)

Test Kasus 1: Akun Influencer Normal
Skor Indikasi Bot: 1.25%
✅ KESIMPULAN: Akun ini kemungkinan besar ASLI (Manusia).
--------------------------------------------------
Test Kasus 2: Akun Bot Giveaway/Spam


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Skor Indikasi Bot: 100.00%
🚨 KESIMPULAN: Akun ini kemungkinan besar PALSU/BOT!
--------------------------------------------------
